# Count matrices

Load, align, and validate the gene-level Salmon matrices.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

FRACTIONS = ('Input', 'Monosome', 'Polysome')
PREFIXES = {
    'Input': 'input',
    'Monosome': 'mono',
    'Polysome': 'poly',
}
FILES = {
    'counts': 'salmon.merged.gene_counts.tsv',
    'lengthScaledCounts': 'salmon.merged.gene_counts_length_scaled.tsv',
    'lengths': 'salmon.merged.gene_lengths.tsv',
    'tpm': 'salmon.merged.gene_tpm.tsv',
}


In [2]:
# Locate the project directory and load sample identifiers.


def find_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / 'data').is_dir():
            return path
    raise FileNotFoundError('Could not find the project data directory.')


ROOT = find_root(Path.cwd().resolve())
DATA_DIR = ROOT / 'data'
metadata = pd.read_csv(DATA_DIR / 'metadata.tsv', sep = '\t', dtype = 'string')
sample_ids = metadata['sampleId'].tolist()


In [3]:
# Read one matrix and retain samples listed in the metadata.


def load_matrix(path: Path, prefix: str) -> tuple[pd.DataFrame, pd.Series]:
    table = pd.read_csv(path, sep = '\t')
    sample_columns = [f'{prefix}{sample_id}' for sample_id in sample_ids]
    required = {'gene_id', 'gene_name', *sample_columns}
    missing = required.difference(table.columns)

    if missing:
        raise ValueError(f'{path.name} is missing columns: {sorted(missing)}')
    if table['gene_id'].isna().any() or not table['gene_id'].is_unique:
        raise ValueError(f'{path.name} contains invalid gene identifiers.')
    if table['gene_name'].isna().any():
        raise ValueError(f'{path.name} contains missing gene names.')

    matrix = table.set_index('gene_id').loc[:, sample_columns]
    column_map = dict(zip(sample_columns, sample_ids, strict = True))
    matrix = matrix.rename(columns = column_map)
    matrix = matrix.apply(pd.to_numeric, errors = 'raise')
    matrix.index.name = 'geneId'
    matrix.columns.name = 'sampleId'

    genes = table.set_index('gene_id')['gene_name'].rename('geneName')
    genes.index.name = 'geneId'

    return matrix, genes


In [4]:
# Load every gene matrix and confirm a common feature space.

matrices = {name: {} for name in FILES}
annotations = {}

for name, filename in FILES.items():
    for fraction in FRACTIONS:
        path = DATA_DIR / fraction / filename
        matrix, genes = load_matrix(path, PREFIXES[fraction])
        matrices[name][fraction] = matrix
        annotations[(name, fraction)] = genes

gene_ids = matrices['counts']['Input'].index
gene_names = annotations[('counts', 'Input')]

for name in FILES:
    for fraction in FRACTIONS:
        matrix = matrices[name][fraction]
        genes = annotations[(name, fraction)]

        if not matrix.index.equals(gene_ids):
            raise ValueError(f'{name} gene order differs in {fraction}.')
        if not genes.equals(gene_names):
            raise ValueError(f'{name} gene annotations differ in {fraction}.')
        if not np.isfinite(matrix.to_numpy()).all():
            raise ValueError(f'{name} contains nonfinite values in {fraction}.')
        if matrix.lt(0).any().any():
            raise ValueError(f'{name} contains negative values in {fraction}.')

for fraction in FRACTIONS:
    if not np.allclose(tpm_sum := matrices['tpm'][fraction].sum(), 1e6):
        raise ValueError(f'TPM does not sum to one million in {fraction}: {tpm_sum}')


In [5]:
# Name the validated objects used by downstream analyses.
# Preserve fractional Salmon estimates until the modeling notebook.

counts = matrices['counts']
length_scaled_counts = matrices['lengthScaledCounts']
lengths = matrices['lengths']
tpm = matrices['tpm']
genes = gene_names.to_frame()

summary = pd.DataFrame([
    {
        'fraction': fraction,
        'geneCount': len(gene_ids),
        'sampleCount': len(sample_ids),
        'medianLibrarySize': counts[fraction].sum().median(),
        'medianDetectedGenes': counts[fraction].gt(0).sum().median(),
        'medianTpmSum': tpm[fraction].sum().median(),
    }
    for fraction in FRACTIONS
])

display(summary)


,fraction,geneCount,sampleCount,medianLibrarySize,medianDetectedGenes,medianTpmSum
0,Input,78941,6,8.010697e+07,29741.5,1.000000e+06
1,Monosome,78941,6,8.141800e+07,25806.0,1.000000e+06
2,Polysome,78941,6,8.238972e+07,26294.0,1.000000e+06
